In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

import anndata
import scanpy as sc

from scipy.stats import mannwhitneyu
from sklearn.cluster import KMeans

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Fail1', 'Delayed', 'Inter1', 'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/raw/mct.passqc_nuclei.h5ad')

In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [ ]:
ips_clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/label_transfer/snm3c_ips_subtype.csv'
ips_clusters = pd.read_csv(ips_clusters_path, sep='\t', header=0, index_col=0)
ips_clusters.shape

In [ ]:
clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna/csv/label_transfer/snmct_xgboost_v7.csv'
clusters = pd.read_csv(clusters_path, sep='\t', header=0, index_col=0)
clusters.shape

In [ ]:
adata.obs['cluster'] = clusters.reindex(adata.obs.index)['pred_coarse']
adata.obs['ips_subtype'] = ips_clusters.reindex(adata.obs.index)['ips_subtype']
ips_adata = adata[~adata.obs['ips_subtype'].isna()]
ips_adata.shape

### check traj_score

In [ ]:
traj_score_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna/csv/label_transfer/snmct_traj_score.csv'
traj_score = pd.read_csv(traj_score_path, sep='\t', index_col=0)
traj_score.shape

In [ ]:
traj_score['ips_subtype'] = adata.obs.reindex(traj_score.index)['ips_subtype']
tmp_traj_score = traj_score[~traj_score['ips_subtype'].isna()]

In [ ]:
traj_score.head()

In [ ]:
ips_traj = traj_score[~traj_score['ips_subtype'].isna()].copy()
ips_traj.shape

In [ ]:
ips_traj.head()

In [ ]:
fig, axes = plt.subplots(1, figsize=(7, 6))

x = 'donor'
y = 'traj_score'
hue = 'ips_subtype'

ax = sns.boxplot(data=ips_traj, x=x, y=y, hue=hue, showfliers=False)
ax.set_xlabel('', fontsize=16)
ax.set_ylabel('Trajectory score', fontsize=16)
ax.set_title('RNA trajectory score', fontsize=20)
ax.grid(False)
ax.tick_params(axis='x', labelsize=12, labelrotation=45)
ax.tick_params(axis='y', labelsize=12, labelrotation=0)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
fig, axes = plt.subplots(1, figsize=(7, 6))

x = 'ips_subtype'
y = 'traj_score'
hue = 'donor'

ax = sns.boxplot(data=ips_traj, x=x, y=y, showfliers=False)
ax.set_xlabel('', fontsize=8)
ax.set_ylabel('Traj score', fontsize=8)
ax.set_title('RNA trajectory score', fontsize=20)
ax.grid(False)
ax.tick_params(axis='x', labelsize=8, labelrotation=45)
ax.tick_params(axis='y', labelsize=8, labelrotation=0)

In [ ]:
for d in donors:
    donor_traj = ips_traj[ips_traj['donor'] == d]
    tmp_main = donor_traj[donor_traj['ips_subtype'] == 'IPS_main']
    tmp_alt = donor_traj[donor_traj['ips_subtype'] == 'IPS_alt']
    a = tmp_main['traj_score']
    b = tmp_alt['traj_score']
    t = mannwhitneyu(a, b)
    print(t)

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure5'
ips_traj.to_csv(f'{indir}/ips_subtype_rna_traj_score.csv', sep='\t')

### check justin's DEGs

In [ ]:
cutoff = 0.01
degs_path = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/csv/degs/justin_degs_v7.csv'
degs = pd.read_csv(degs_path, sep='\t')
degs = degs[degs['p_val_adj'] < cutoff]
degs.shape

In [ ]:
degs.head()

In [ ]:
n = 20
tmp_clusts = ['IPS_1', 'IPS_2', 'IPS_3']
for s in tmp_clusts:
    mask = degs['cluster'] == s
    tmp_degs = degs[mask].sort_values(by=['avg_log2FC'])
    tmp_genes = pd.Index(tmp_degs['gene'][:n].values)
    final_genes = list(tmp_genes[tmp_genes.isin(adata.var.index)])
    sc.tl.score_genes(ips_adata, gene_list=final_genes, score_name=s)

In [ ]:
tmp_meta = ips_adata.obs.copy()
tmp_meta.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8,8))
for i, ax in enumerate(axes.flatten()):
    if i > 2: break
    s = tmp_clusts[i]
    sns.boxplot(tmp_meta, x='donor', y=s, hue='ips_subtype', ax=ax, showfliers=False)
    ax.grid(False)
    ax.set_title(s)
    
plt.subplots_adjust(wspace=0.5, hspace=0.5)

In [ ]:
n = 50
tmp_clusts = ['IPS_1', 'IPS_2', 'IPS_3']
for s in tmp_clusts:
    mask = degs['cluster'] == s
    tmp_degs = degs[mask].sort_values(by=['avg_log2FC'])
    tmp_genes = pd.Index(tmp_degs['gene'][:n].values)
    final_genes = list(tmp_genes[tmp_genes.isin(adata.var.index)])
    sc.tl.score_genes(ips_adata, gene_list=final_genes, score_name=s)
tmp_meta = ips_adata.obs.copy()
tmp_meta.head()

fig, axes = plt.subplots(2, 2, figsize=(8,8))
for i, ax in enumerate(axes.flatten()):
    if i > 2: break
    s = tmp_clusts[i]
    sns.boxplot(tmp_meta, x='donor', y=s, hue='ips_subtype', ax=ax, showfliers=False)
    ax.grid(False)
    ax.set_title(s)
    
plt.subplots_adjust(wspace=0.5, hspace=0.5)

### let's grab rev class

In [ ]:
rev_path = '/u/project/cluo/Shared_Datasets/IGVF/202208_Pilot/new_multiome/YS3v7_Data/CL1.metadata.with.ys3_v7_revclass.txt'
rev = pd.read_csv(rev_path, sep='\t')
rev.shape

In [ ]:
rev.head()

In [ ]:
ips_traj['rev_class'] = rev.reindex(ips_traj.index)['RevClass2']
ips_traj = ips_traj.dropna()
ips_traj.head()

In [ ]:
tmp_clusts = ips_traj['ips_subtype'].unique()
rev_clusts = ips_traj['rev_class'].unique()

rev_counts = pd.DataFrame(index=rev_clusts)

for s in tmp_clusts:
    tmp_rev = ips_traj[ips_traj['ips_subtype'] == s]
    rev_counts[s] = tmp_rev['rev_class'].value_counts()
    
rev_counts = rev_counts.fillna(0).astype(int)
rev_counts.head()

In [ ]:
rev_dist = rev_counts.div(rev_counts.sum(axis=0), axis=1)
rev_dist.sum(axis=0)

In [ ]:
rev_dist

In [ ]:
ax = rev_dist.T.plot(kind="bar", stacked=True, figsize=(8,5))
ax.set_ylabel("Proportion")
ax.set_xlabel("")
ax.set_title("Cell state proportions: IPS_main vs IPS_alt")
ax.grid(False)
plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()

### score IPS_1, IPS_2, and IPS_3 genes

In [ ]:
ips_adata

In [ ]:
s = 'IPS_1'
n = 50
tmp_clusts = ['IPS_1', 'IPS_2', 'IPS_3']

for s in tmp_clusts:
    tmp_degs = degs[degs['cluster_v7'] == s].sort_values(by=['avg_log2FC'], ascending=False)
    tmp_genes = tmp_degs['gene'][:n].values
    sc.tl.score_genes(ips_adata, gene_list = tmp_genes, score_name=s)

In [ ]:
tmp_meta = ips_adata.obs.copy().sort_values(by=['ips_subtype'])
tmp_meta.head()

In [ ]:
to_plot = tmp_meta[tmp_clusts].copy()

k = 6
kmeans = KMeans(n_clusters=k)
kmeans.fit(to_plot)
k_clusters = kmeans.predict(to_plot)
to_plot['cluster'] = k_clusters
cluster_sizes = to_plot['cluster'].value_counts().sort_index().values
to_plot = to_plot.sort_values(by=['cluster']).iloc[:,:3]

In [ ]:
s = 'IPS_main'
tmp_labels = ips_adata.obs.reindex(to_plot.index)['ips_subtype']
mask = tmp_labels == s
main_plot = to_plot[mask].copy()
kmeans = KMeans(n_clusters=k)
kmeans.fit(main_plot)
k_clusters = kmeans.predict(main_plot)
main_plot['cluster'] = k_clusters
cluster_sizes = main_plot['cluster'].value_counts().sort_index().values
main_plot = main_plot.sort_values(by=['cluster']).iloc[:,:3]
sns.heatmap(main_plot, cmap='coolwarm', vmin=-0.5, vmax=0.5)

In [ ]:
s = 'IPS_alt'
tmp_labels = ips_adata.obs.reindex(to_plot.index)['ips_subtype']
mask = tmp_labels == s
alt_plot = to_plot[mask].copy()
kmeans = KMeans(n_clusters=k)
kmeans.fit(alt_plot)
k_clusters = kmeans.predict(alt_plot)
alt_plot['cluster'] = k_clusters
cluster_sizes = alt_plot['cluster'].value_counts().sort_index().values
alt_plot = alt_plot.sort_values(by=['cluster']).iloc[:,:3]
sns.heatmap(alt_plot, cmap='coolwarm', vmin=-0.5, vmax=0.5)

### decide what to write out


In [ ]:
!date